# Sistemas de Memoria para Validacion de Guiones

Este notebook demuestra tres estrategias de memoria aplicadas al agente de continuidad:
1. **ConversationBufferMemory** — Historial completo de corto plazo.
2. **ConversationBufferWindowMemory** — Ventana deslizante de ultimas N interacciones.
3. **ConversationSummaryMemory** — Resumen automatico para conversaciones prolongadas.

Cada estrategia se ajusta a distintos escenarios de validacion de guiones.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
)
from langchain_core.messages import HumanMessage, AIMessage

from transformers_rag import ContinuityConsultant, ContinuityReport

## Configuracion del LLM para operaciones de memoria

In [ ]:
def get_llm(temperature: float = 0.0):
    return ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        temperature=temperature,
    )

## Estrategia 1: ConversationBufferMemory (Historial completo)

Mantiene TODO el historial de validaciones en memoria. Ideal para sesiones cortas donde el contexto completo es necesario.

In [ ]:
class BufferMemoryValidator:

    def __init__(self):
        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="output",
        )
        self.consultant = ContinuityConsultant()
        self.consultant.initialize()

    def validate(self, script_fragment: str) -> ContinuityReport:
        report = self.consultant.analyze(script_fragment)
        self.memory.save_context(
            {"input": f"Analiza este fragmento: {script_fragment}"},
            {"output": f"Veredicto: {report.verdict}, Confianza: {report.confidence_score:.0%}"}
        )
        return report

    def ask_followup(self, question: str) -> str:
        history = self.memory.load_memory_variables({})["chat_history"]
        context_lines = []
        for msg in history:
            role = "Usuario" if isinstance(msg, HumanMessage) else "Sistema"
            context_lines.append(f"{role}: {msg.content}")
        context = "\n".join(context_lines)

        llm = get_llm()
        response = llm.invoke(
            f"Basandote en el siguiente historial de validaciones, responde la pregunta.\n\n"
            f"HISTORIAL:\n{context}\n\n"
            f"PREGUNTA: {question}\n\n"
            f"Responde de forma concisa y tecnica."
        )
        return response.content

## Estrategia 2: ConversationBufferWindowMemory (Ventana deslizante)

Mantiene solo las ultimas N interacciones. Ideal para flujos prolongados donde el contexto remoto es irrelevante y se desea ahorrar tokens.

In [ ]:
class WindowMemoryValidator:

    def __init__(self, k: int = 3):
        self.memory = ConversationBufferWindowMemory(
            k=k,
            memory_key="chat_history",
            return_messages=True,
            output_key="output",
        )
        self.consultant = ContinuityConsultant()
        self.consultant.initialize()
        self.k = k

    def validate(self, script_fragment: str) -> ContinuityReport:
        report = self.consultant.analyze(script_fragment)
        self.memory.save_context(
            {"input": f"Fragmento: {script_fragment[:80]}..."},
            {"output": f"Veredicto={report.verdict}, Confianza={report.confidence_score:.0%}"}
        )
        return report

    def ask_followup(self, question: str) -> str:
        history = self.memory.load_memory_variables({})["chat_history"]
        context_lines = []
        for msg in history:
            role = "Usuario" if isinstance(msg, HumanMessage) else "Sistema"
            context_lines.append(f"{role}: {msg.content}")
        context = "\n".join(context_lines)

        llm = get_llm()
        response = llm.invoke(
            f"Contexto reciente (ultimas {self.k} interacciones):\n"
            f"{context}\n\nPregunta: {question}"
        )
        return response.content

## Estrategia 3: ConversationSummaryMemory (Resumen para largo plazo)

Resume el historial automaticamente cuando la conversacion crece. Ideal para sesiones muy largas donde el historial completo excederia el limite de contexto.

In [ ]:
class SummaryMemoryValidator:

    def __init__(self):
        self.llm = get_llm()
        self.memory = ConversationSummaryMemory(
            llm=self.llm,
            memory_key="chat_history",
            return_messages=True,
            output_key="output",
        )
        self.consultant = ContinuityConsultant()
        self.consultant.initialize()

    def validate(self, script_fragment: str) -> ContinuityReport:
        report = self.consultant.analyze(script_fragment)
        self.memory.save_context(
            {"input": f"Validar fragmento: {script_fragment[:80]}..."},
            {"output": f"Resultado: {report.verdict} (confianza {report.confidence_score:.0%})"}
        )
        return report

    def get_summary(self) -> str:
        variables = self.memory.load_memory_variables({})
        return variables.get("chat_history", "Sin resumen disponible.")

    def ask_followup(self, question: str) -> str:
        summary = self.get_summary()
        response = self.llm.invoke(
            f"Resumen de validaciones realizadas:\n{summary}\n\n"
            f"Pregunta del usuario: {question}\n\n"
            f"Responde basandote en el resumen."
        )
        return response.content

## Demostracion comparativa

Tres fragmentos de prueba para comparar las estrategias de memoria.

In [ ]:
print("="*70)
print(" DEMO: Sistemas de Memoria para Validacion de Guiones")
print("="*70)

fragment_1 = (
    "Escena 5 — 2007. Bumblebee mira a Sam y dice en voz clara: '
    "'Sam, debes proteger el AllSpark a toda costa.'"
)
fragment_2 = (
    "Escena 8 — 2011. Ironhide lidera la defensa de Chicago contra los Decepticons."
)
fragment_3 = (
    "Escena 12 — 2014. Cade Yeager encuentra a Optimus Prime escondido en Texas."
)

In [ ]:
print("\n>>> Estrategia 1: ConversationBufferMemory (historial completo)")
buffer_val = BufferMemoryValidator()
r1 = buffer_val.validate(fragment_1)
print(f"  Fragmento 1 -> {r1.verdict} (confianza {r1.confidence_score:.0%})")
r2 = buffer_val.validate(fragment_2)
print(f"  Fragmento 2 -> {r2.verdict} (confianza {r2.confidence_score:.0%})")

print("\n  [Usuario] Que inconsistencias detectaste en total?")
resp = buffer_val.ask_followup("Que inconsistencias detectaste en total?")
print(f"  [Agente] {resp}")

In [ ]:
print("\n>>> Estrategia 2: ConversationBufferWindowMemory (k=2)")
window_val = WindowMemoryValidator(k=2)
window_val.validate(fragment_1)
window_val.validate(fragment_2)
window_val.validate(fragment_3)
print(f"  Conserva las ultimas 2 interacciones para ahorrar tokens.")

In [ ]:
print("\n>>> Estrategia 3: ConversationSummaryMemory (resumen automatico)")
summary_val = SummaryMemoryValidator()
summary_val.validate(fragment_1)
summary_val.validate(fragment_2)
summary_val.validate(fragment_3)
print(f"  Resumen de memoria:")
print(f"  {summary_val.get_summary()}")

print("\n" + "="*70)